# 专利数据审计

本 Notebook 只读取 raw，先完成结构、格式、数值、重复和财务覆盖审计。

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')
import pandas as pd

from src.patent_cleaning import PATENT_VALUE_COLUMNS, audit_patents, normalize_stock_code

raw = pd.read_csv(Path('../data/raw/patents.csv'), dtype=object, keep_default_na=False)
financial = pd.read_parquet(Path('../data/processed/firm_financials_clean.parquet'))
raw.head()

,stock_code,year,invention_patents,utility_patents,patent_citations
0,000001,2020,8.0,19.0,33.0
1,000001,2021,8.0,16.0,23.0
2,000001,2023,8.0,9.0,24.0
3,1,2024,10.0,8.0,29.0
4,000001,2025,12.0,7.0,39.0


In [2]:
audit_patents(raw, financial)

{'raw_rows': 225,
 'raw_columns': 5,
 'raw_firms': 40,
 'year_min': 2020,
 'year_max': 2025,
 'unique_firm_year': 220,
 'duplicate_member_rows': 10,
 'duplicate_groups': 5,
 'financial_firm_year': 240,
 'observed_financial_firm_year': 220,
 'unobserved_financial_firm_year': 20,
 'patent_firms': 40,
 'patent_value_missing': {'invention_patents': 1,
  'utility_patents': 1,
  'patent_citations': 1}}

In [3]:
candidate = normalize_stock_code(raw['stock_code'])
audit_key = pd.DataFrame({'stock_code': candidate, 'year': pd.to_numeric(raw['year'])})
duplicate_rows = audit_key.duplicated(['stock_code', 'year'], keep=False)
duplicate_detail = raw.loc[duplicate_rows].assign(
    stock_code_candidate=candidate[duplicate_rows].to_numpy()
)
duplicate_detail.sort_values(['stock_code_candidate', 'year'])

,stock_code,year,invention_patents,utility_patents,patent_citations,stock_code_candidate
4,000001,2025,12.0,7.0,39.0,000001
220,000001,2025,12.0,7.0,39.0,000001
33,000006,2025,10.0,13.0,34.0,000006
221,000006,2025,10.0,13.0,34.0,000006
72,000014,2020,8.0,23.0,27.0,000014
222,000014,2020,8.0,23.0,27.0,000014
111,000020,2025,7.0,14.0,27.0,000020
223,000020,2025,7.0,14.0,27.0,000020
150,000028,2021,7.0,9.0,17.0,000028
224,000028,2021,7.0,9.0,1234.0,000028


In [4]:
numeric_audit = []
for column in PATENT_VALUE_COLUMNS:
    values = pd.to_numeric(raw[column], errors='coerce')
    quantiles = values.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    numeric_audit.append({
        'variable': column,
        'missing': int(values.isna().sum()),
        **quantiles.to_dict(),
        'max': values.max(),
    })
pd.DataFrame(numeric_audit)

,variable,missing,0.01,0.05,0.25,0.5,0.75,0.95,0.99,max
0,invention_patents,1,2.23,3.15,6.0,7.0,10.0,12.0,14.77,16.0
1,utility_patents,1,6.00,7.00,10.0,13.0,16.0,20.0,24.00,25.0
2,patent_citations,1,6.23,10.00,16.0,22.0,29.0,41.0,62.39,99999.0


## 审计结论

原始表 225 行、40 家企业、220 个唯一 firm-year；5 个重复组、10 条重复组成员，其中 4 组完全重复、1 组引用数冲突。三个专利字段各有 1 条原始缺失。引用数 99,999 和冲突版本 1,234 明显脱离主体分布，正式清洗时依据稳健分布规则处理。财务面板中未观测到的 20 个 firm-year 不补为零。